# model_a — TABAN koşusu

Önceden kayıt: `belge/onkayit/model_a.md`

`SABLON.ipynb`'in kopyası. Tek fark: aşağıdaki `MODEL` satırı.

Bu koşu tez sınamıyor — **zemini ölçüyor**: doyma adımı, ENT seviyesi,
kısayol oranı, en iyi pencere. Bütçe 80.000 (CLAUDE.md kural 1).

In [ ]:
MODEL = "model_a"          # <-- DEGISTIRILECEK TEK SATIR

assert MODEL, ("MODEL bos. Bu SABLON -- kopyala, adini modelin adi yap "
               "(model_a.ipynb) ve bu satiri doldur.")
DEPO = "https://github.com/sekerahmet/sekerai.git"
KOD  = "/content/kod"
EV   = f"/content/drive/MyDrive/deneme2/{MODEL}"
LOG  = f"{EV}/log.txt"
print(MODEL, "->", EV)

## 1 — GPU

CPU'ya düşerse 45 dakikalık iş **36 saat** olur (ölçüldü: CPU'da adım başına
1,64 sn). Burada durmak, 36 saat sonra fark etmekten iyidir.

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU YOK -- Calisma zamani > Turu degistir > T4"
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

## 2 — Drive

Çıktı doğrudan Drive'a yazılır. `/content` runtime ölünce silinir, Drive silinmez.

In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive")
os.makedirs(EV, exist_ok=True)
print("hazir:", EV)

## 3 — Kod

**Colab'da yama yok.** Depo silinip yeniden klonlanır, üzerine hiçbir şey
yazılmaz. Tek kaynak: GitHub.

Yereli düzeltip itmeyi unuttuysan burada görürsün — klon eski commit'i getirir.

In [ ]:
import subprocess, os, importlib, sys
subprocess.run(["rm","-rf",KOD], check=True)
subprocess.run(["git","clone","--depth","1",DEPO,KOD], check=True)
commit = subprocess.run(["git","-C",KOD,"log","-1","--format=%h %s"],
                        capture_output=True, text=True).stdout.strip()
print("commit:", commit)

sys.path.insert(0, f"{KOD}/deneme2")
M = importlib.import_module(MODEL)
print("ayar:", M.AYAR)

## 4 — Başlat

**Ayrı süreç.** Çekirdek serbest kalır: ilerlemeye bakabilir, durdurabilirsin.
Hücreye gömülürse çekirdek kilitlenir ve koşu bitene kadar hiçbir şey
yapamazsın.

In [ ]:
kos = f"""
import sys; sys.path.insert(0, "{KOD}/deneme2")
import {MODEL} as M
M.egit(M.AYAR, alt="{EV}")
"""
open("/content/kos.py","w").write(kos)
p = subprocess.Popen([sys.executable,"-u","/content/kos.py"],
                     stdout=open(LOG,"w"), stderr=subprocess.STDOUT)
print("PID", p.pid, "-> log:", LOG)

## 5 — İlerleme

Bu hücre **hesap yapmaz, GPU kullanmaz** — sadece dosya okur. Koşu
kilitliyken bile çalışır. İstediğin kadar tekrar çalıştır.

In [ ]:
print(open(LOG).read()[-3000:])

## 6 — Durdurmak

`p.kill()` — ya da runtime'ı kapat. Anlık görüntüler Drive'da kalır.

**Sürdürme yok.** Koşu kesilirse baştan başlar (~45 dk). 80.000 adım için
bu kabul edilebilir; olmadığı gün sürdürme eklenir.

In [ ]:
# p.kill()